# 4b · Multiple-instance training

Stage 5 of the chain. `4a` trains one cell at a time; this notebook trains one **cell line** at a
time and asks what the per-cell architecture structurally cannot: does the model give the cells of a
line *different* predicted responses?

**Position** `3_representations` → `4a_percell_training` → **`4b`** → `5_evaluation`
Reads one of `4a`'s outputs, so the order is a hard edge rather than a preference.

| | |
|---|---|
| **reads** | `..._with_targets_auc_cc.h5ad` — 53,513 cells × {`X_pca` 512, `X_scGPT` 512}, `Y_ctrp`, `M_ctrp`, `split_ctrp`, QC covariates |
| | `outputs/panel/panel.csv` — 11 drugs, the column selection |
| | `outputs/panel/panel_within_line_spread.csv` — `4a`'s within-line spread, stage 1's reference |
| **writes** | `outputs/mil/stage{0,7,1,2,6}_*.csv` — one table per criterion stage |
| | `outputs/mil/q2_verdict.csv`, `mil_oof_predictions.csv`, `mil_within_line_spread.csv` |
| | `runs/percell/percell_mil_<rep>_a<α>_<loss>_s<seed>.npy` — `(53513, 11)` float32 per run |

**Architecture**

```
one bag = all cells of one cell line
  (n_cells, 512)  embedding, X_pca or X_scGPT
       ↓ shared trunk    OncoMLP (128, 64), LayerNorm, GELU, dropout 0.5 — identical to 4a
       ↓ Linear(→ 11)
  (n_cells, 11)   per-cell predicted response  ← the native output; every stage below reads it
       ↓ mean over the bag's cells             ← the only change from 4a
  (11,)           bag prediction
       ↓ masked MSE against the line's label vector and its mask
```

Encoder, target, splits and optimiser are `4a`'s, unchanged. Aggregation is the single moving part,
so a difference between the two notebooks is attributable to it.

**Run scope** 2 representations × 3 seeds, α = 0.5, `loss = mse`, full-line bags, lines weighted
equally. Six runs of five folds. The representation is part of the question, not a setting: one
encoding carrying within-line structure while the other does not is a result about the encoding.

**Why the per-cell model cannot answer this.** For one line with predictions $p_c$, mean $\bar p$
and label $y$:

$$\tfrac{1}{n}\textstyle\sum_c (p_c-y)^2 \;=\; (\bar p-y)^2 \;+\; \mathrm{Var}_c(p)$$

The left side is `4a`'s objective regrouped by line; the first term on the right is what mean pooling
minimises. They differ by exactly the within-line variance of the predictions, at full weight, in
every batch. The per-cell objective therefore *penalises* the quantity under study. It is an
identity, not a tendency, so no learning rate, capacity or regulariser reaches it — only removing the
term does.

**Why bags are complete.** For a sub-bag of $B$ cells drawn from a line of $n$ the expected loss is
$(\bar p_n-y)^2+(\sigma^2/B)(1-\tfrac{B-1}{n-1})$: at $B=1$ exactly `4a`'s loss, at $B=n$ no variance
term at all. Bag size interpolates between the two architectures rather than tuning either. Complete
bags also keep one line as one training example, which removes the per-cell objective's implicit
weighting of lines by sequencing depth. Cost: one gradient step per line, and memory scaling with the
largest line (1,990 cells).

**Why instance-level rather than attention pooling.** Every cell receives its own predicted response
and the bag averages them, rather than every cell receiving an attention weight over a pooled
embedding (Ilse, Tomczak & Welling, ICML 2018). Embedding-level generally predicts better;
instance-level is readable at the resolution the question is asked at, and that trade is taken
deliberately.

One consequence, so it is not rediscovered: selecting "the top-k cells" by predicted value and
scoring them against the line's response is biased by construction, because the extremes were chosen
for being extreme. No stage below does it.

**Why mean pooling, and why it is not revisited.** The bag prediction is the mean of its cells'
predictions, matching the definition of the label it is compared against. A variance term in the loss
would make stage 1 pass by construction; a max or top-quantile aggregator does not force variation
either, since the maximum of equal values is that value. Mean pooling carries a shrinkage incentive —
collapsing a bag onto its mean is a minimiser whenever the model cannot do better — which is why the
criterion leads with a positive control. If that control fails the aggregator is **not** exchanged
for one that passes: adjusting an instrument until it certifies itself is a forking path moved down
one level.

**What a result can and cannot establish.** Reachable: whether the variation exists, reproduces
across seeds, and survives the confound check. Out of reach: whether it is real heterogeneity of drug
response, and whether it identifies the cells that survive treatment — no per-cell response was ever
measured, and no primary source carries post-treatment single-cell profiles. The first is a necessary
condition for the second, so a negative is decisive and a positive earns a hypothesis, not a finding.

§2 fixes the criterion before any model exists, and it is not revised after seeing results.

## 2 · What counts as a positive result — fixed before the run

Settled before any model existed. Every bar below is a permutation null, a comparison against `4a`,
or a collapse test: the criterion contains no magnitude chosen by judgement.

That was not true of the first draft, which carried three invented numbers — a floor on stage 0, an
AUROC bar on stage 7, a fraction on stage 2. Each was removed by replacing the bar with a null or a
comparison, not by picking a better number. Stage 7's mattered most: a failure there ends the run, so
an arbitrary bar decided whether the project reported anything at all.

There is no ground truth for within-line heterogeneity of drug response. Every label is one number
per (cell line, drug), so the ideal test — do the model's resistant cells match the cells that
survive treatment — cannot be run. What can be established is narrower: that the predictions vary
within a line, that the variation is reproducible rather than noise, and that it is not a sequencing
artifact.

| # | stage | role | passes when |
|---|---|---|---|
| **0** | within-line dispersion of the representation, before training | precondition on the **input** | a line's cells are not collapsed to a point |
| **7** | synthetic positive control, scored **inside** a mixed bag | precondition on the **instrument** | within-bag rank separation beats its permutation null |
| **1** | within-line sd of per-cell predictions | necessary condition | it exceeds `4a`'s, same lines, drugs, folds, seed |
| **2** | do independent seeds order the same cells alike | **the test** | per-cell agreement beats the shuffled-cell null |
| **6** | predictions regressed on depth, genes, mito, cell cycle | **veto** | confounds explain less than the signal reproduces |

**Why stage 0 first.** It costs no training, and it separates *the input carries no structure* from
*the model did not use it*. Asked per representation, and it may pass for one and fail for the other.

**Why stage 7 before the rest.** Without it a negative is uninterpretable — "no heterogeneity found"
cannot be distinguished from "this method cannot find heterogeneity". With it, a negative becomes a
result at a measured sensitivity, reported alongside it.

**Why stage 7 is scored inside the bag.** The bag prediction is a mean, so a model assigning every
cell in a bag the same value lands the bag label exactly while doing none of what stages 1, 2 and 6
measure. A bag-level control would certify a collapsed instrument. Bag-level recovery is still
computed across mixture weights and reported, but it cannot pass the stage.

**Why stage 6 is a veto and not an analysis.** Predictions that replicate across seeds *and* are
explained by library size are a sequencing artifact, not biology — the confounds themselves
reproduce. Pre-registered so it cannot become something run only when the answer is unwelcome.

### 2.1 · Stage 7 — the control

For each drug, take line pairs whose measured responses differ by an amount in the **bottom quartile**
of that drug's pairwise $|y_A-y_B|$, treat the union of their cells as one bag mixed at weight 0.5,
and ask whether A-cells rank above B-cells *within* that bag. The statistic is the within-bag AUROC —
the Mann–Whitney $U/(n_A n_B)$ — and it passes by beating the null from permuting which cells came
from which source, which is that statistic's own null.

**Why close pairs, not far apart.** The only response differences ever measured here are *between*
lines, so any manufactured control inherits a between-line magnitude, larger than any plausible
within-line heterogeneity. Pairing on a large gap gives a control that is easy to pass and useless
about the regime the question operates in. The cost is taken deliberately: the control is harder, and
a failure ends the run.

**Why ordering, not recovered magnitude.** The recovered gap fraction
$(\bar p_A - \bar p_B)/(y_A - y_B)$ is on the label's own scale but is calibration-sensitive, and mean
pooling gives the model a standing shrinkage incentive — so a model that orders correctly but pulls
toward the bag mean would fail for the same reason stage 1 would, costing stage 7 its independence
from the stage it licenses. Its denominator is also small by construction under bottom-quartile
pairing. AUROC reads only order, which is what stages 1 and 2 rest on. The gap fraction is still
computed and reported as a description.

### 2.2 · Stage 1 — spread, against `4a` rather than a number

Passes when MIL's within-line sd of per-cell predictions **exceeds `4a`'s** on the same lines, drugs,
folds, representation and seed. No margin. The identity in §1 shows why none is needed: `4a`'s
objective charges for within-line variance at full weight and the bag objective does not contain the
term, so `4a`'s spread is spread that survived an explicit penalty and MIL's is spread with the
penalty removed. Comparing them tests precisely the deleted term.

This makes `4a` a hard predecessor. Note that `pred_std` in `panel_per_drug_correlation.csv` is the
spread of *line-level* predictions **across** lines — a between-line quantity, the opposite of what
stage 1 needs.

### 2.3 · Stage 2 — reproducibility, the test

Passes when per-cell agreement across independent seeds beats the shuffled-cell null: the same
statistic after permuting cell identities within each line, which destroys cell-specific signal while
preserving each seed's marginal distribution. Three seeds — the minimum that makes "the same cells
under different initialisations" a comparison. The agreement value is reported, so a result that is
statistically clear but small is visible as such.

### 2.4 · Stage 6 — the confound veto

Regress each cell's predicted response on total counts, genes detected, mitochondrial fraction and
cell-cycle score, within line.

The veto fires when the confounds explain **as much of the within-line variation as the signal
reproduces** — stage 6's adjusted R² against stage 2's cross-seed agreement, put on one scale by
squaring ρ. A permutation null cannot supply this magnitude: with hundreds of cells per line, an R²
far too small to matter is still significant against one. It is the single bar in §2 a null cannot
replace, so it is settled by comparison against something the run measures rather than by a constant.

### 2.5 · Run scope

`pca` and `scgpt`, both, separately · α = 0.5 only, `4a`'s default · three seeds · full-line bags,
lines weighted equally. Two runs of three seeds. `4a` must have run first.

α is not swept here: the architecture is the change under test, and moving two things at once makes
the difference unattributable.

### 2.6 · What each outcome will be written as

Fixed here for the same reason the bars are: a number does not determine a claim on its own. §2
already did this once — a stage-7 failure means *"Q2 unanswered, the instrument was not
demonstrated"*, never *"no heterogeneity found"* — and the rest of the write-up had the same freedom
with none of the discipline.

| outcome | written as | not as |
|---|---|---|
| stage 0 collapses | the representation places a line's cells at one point; no model could separate them | anything about the model, which never ran |
| stage 7 fails | Q2 unanswered, instrument not demonstrated, AUROC given | "no heterogeneity found"; the aggregator is not swapped and retried |
| stage 1 fails | removing the variance penalty did not produce more within-line variation | "there is no heterogeneity" — this is about the objective, not the biology |
| stage 2 fails | the variation does not reproduce; it is initialisation noise | "the model found nothing" — stage 1 may still have passed, and that combination is the finding |
| stage 2 passes but small | ρ and its null in the same sentence, described as distinguishable from noise | a bare "significant" with the magnitude omitted |
| stage 6 vetoes | explained by depth or cell cycle at least as well as it reproduces; a reproducible technical artifact | any softening that leaves the positive standing |
| all pass | Q2(a) positive, at a measured instrument sensitivity of AUROC x.xx | "the model learned heterogeneity" — that is Q2(b), not addressed |

Three rules across all of them. Every negative carries stage 7's AUROC in the same sentence, because
a negative from an instrument of unknown sensitivity is not a result. Magnitude adjectives require a
stated comparison. A split between representations is a result about the encoding, not a nuisance.

## 3 · Running the criterion

Implements §2 and adds nothing to it. One cell per stage, in §2's order — 0, 7, 1, 2, 6 — each
printing the quantity §2 says it reports, whether or not that quantity is also a gate.

The bag model is [`scripts/training/mil.py`](../scripts/training/mil.py), not this notebook. It
imports `grouped_folds` and `inner_holdout` from `cv.py` rather than re-deriving them, so fold *f*
holds out the same cell lines here as in `4a` under every seed. Stage 1 compares the two
architectures on those folds; a second fold implementation is how that premise would quietly stop
holding.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
NB_DIR = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # runs/ lives at the project root

import anndata as ad
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, spearmanr

from scripts.layout import PipelinePaths
from scripts.training.cv import line_level_predictions
from scripts.training.density_weighting import DEFAULT_CAP, line_level
from scripts.training.mil import bag_oof_predictions
from scripts.training.training_utils import TrainConfig

# 4b's own output tree. 4a's `outputs/panel/` is the panel run's; keeping them apart is what lets a
# reader tell which architecture produced a table from its path alone. The one file that crosses
# between them is 4a's within-line spread table, which stage 1 READS and never writes.
OUT = NB_DIR / 'outputs' / 'mil'
OUT.mkdir(parents=True, exist_ok=True)
PERCELL = Path('runs') / 'percell'          # shared with 4a; the filenames carry the architecture
PERCELL.mkdir(parents=True, exist_ok=True)
PANEL_OUT = NB_DIR / 'outputs' / 'panel'

SCORE, VARIANT = 'auc_cc', 'hvg5000'
REPS = ['X_pca', 'X_scGPT']
N_SPLITS = 5

# THE PANEL IS READ, NOT RESTATED -- the same rule 4a follows, for the same reason: a literal here
# would let 4b train on a different drug set than the model it is compared against.
PANEL_CSV = PANEL_OUT / 'panel.csv'
if not PANEL_CSV.exists():
    raise FileNotFoundError(
        f'{PANEL_CSV} not found; it is written by 2_drug_selection.ipynb. There is deliberately no '
        f'fallback list.')
PANEL = pd.read_csv(PANEL_CSV)['drug_key'].tolist()

# alpha = 0.5 ONLY, and it is not swept here (§2.6). 4a sweeps {0, 0.5, 1} because the weighting is
# what that notebook is testing; here the ARCHITECTURE is the change under test, and moving two
# things at once makes the difference unattributable (docs/TODO.md, the governing rule).
ALPHA = 0.5
# Der Loss, mit dem 4b trainiert -- und damit auch der, gegen den Stage 1 vergleicht. 4a faehrt seit
# 13.08.2026 MSE **und** MAE (Item 9A), seine Spread-Tabelle hat also zwei Zeilen pro
# (rep, seed, drug, cell_line). Ohne diesen Filter faende der Merge beide und `validate='one_to_one'`
# wuerde -- korrekterweise -- abbrechen. Stage 1 vergleicht Architekturen, nicht Losses: beide Seiten
# muessen denselben Loss fahren, sonst unterscheidet sich mehr als die Architektur.
LOSS = 'mse'
# Three seeds, because stage 2 asks whether independent initializations rank the same cells alike and
# two points do not make that a comparison. 42 is 4a's; 43 and 44 are its successors and are
# ARBITRARY -- any three distinct integers would do, and nothing in the data picks these.
SEEDS = (42, 43, 44)
EPOCHS = 50  # 4a's, unchanged. See the banner below for what that means at one step per line.

paths = PipelinePaths.build(None, VARIANT, SCORE)
print(f'panel        : {len(PANEL)} drugs from {PANEL_CSV.name}')
print(f'targets      : {paths.targets_h5ad.name}')
print(f'run scope    : {len(REPS)} reps x {len(SEEDS)} seeds at alpha={ALPHA} '
      f'= {len(REPS) * len(SEEDS)} runs of {N_SPLITS} folds')
print(f'weighting    : alpha={ALPHA}, cap={DEFAULT_CAP} (4a defaults; not swept here)')
print(f'epochs       : {EPOCHS} (4a\'s cap). One bag is one gradient step, so an epoch is ~120 steps '
      f'here against ~330 in 4a at batch_size=128 -- same cap, fewer updates.')

panel        : 11 drugs from panel.csv
targets      : SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad
run scope    : 2 reps x 3 seeds at alpha=0.5 = 6 runs of 5 folds
weighting    : alpha=0.5, cap=3.0 (4a defaults; not swept here)
epochs       : 50 (4a's cap). One bag is one gradient step, so an epoch is ~120 steps here against ~330 in 4a at batch_size=128 -- same cap, fewer updates.


### 3.1 · Load

| | |
|---|---|
| **in** | `..._with_targets_auc_cc.h5ad`, read backed |
| **out** | in-memory `AnnData`: `obsm` = both embeddings + `Y_ctrp`/`M_ctrp`, `obs` intact |

The targets file is 2.4 GB, almost all of it the expression matrix `.X`, which no model here touches
— both architectures consume the precomputed embeddings in `obsm`. Reading backed and assembling a
light object keeps `4a` and `4b` on the same array contents as well as the same code.

`uns['ctrp_drugs']` is rewritten to the panel because `build_bags` matches target columns against it,
exactly as `MultiDrugDataset` does.

In [2]:
src = ad.read_h5ad(paths.targets_h5ad, backed='r')
all_drugs = list(src.uns['ctrp_drugs'])
kcol = [all_drugs.index(d) for d in PANEL]

Y_raw = np.asarray(src.obsm['Y_ctrp'], dtype=np.float32)[:, kcol]
M = np.asarray(src.obsm['M_ctrp'], dtype=bool)[:, kcol]
Y = np.where(M, Y_raw, 0.0).astype(np.float32)  # used as published: no clipping (Step 01)

adata = ad.AnnData(obs=src.obs.copy())
for rep in REPS:
    adata.obsm[rep] = np.asarray(src.obsm[rep], dtype=np.float32)
adata.obsm['Y_ctrp'], adata.obsm['M_ctrp'] = Y, M
adata.uns['ctrp_drugs'] = PANEL
src.file.close()

groups = adata.obs['Cell_line'].astype(str).to_numpy()
eligible = adata.obs['split_ctrp'].isin(['train', 'val']).to_numpy()  # fixed test set held out
lines_elig = np.unique(groups[eligible])
n_cells_per_line = pd.Series(groups[eligible]).value_counts()
print(f'{adata.n_obs} cells | {len(lines_elig)} eligible cell lines | K={len(PANEL)} drugs')
# Bag sizes are worth printing rather than assuming: one bag is one line, so this IS the distribution
# of training-example sizes, and the largest line sets peak memory (mil.py, module docstring).
print(f'bag size: min {n_cells_per_line.min()} | median {int(n_cells_per_line.median())} | '
      f'max {n_cells_per_line.max()} cells')

53513 cells | 153 eligible cell lines | K=11 drugs
bag size: min 56 | median 233 | max 1990 cells


### 3.2 · Stage 0 — the input ceiling

| | |
|---|---|
| **in** | `adata.obsm[rep]` for the eligible cells, both representations |
| **out** | `stage0_input_ceiling.csv` — one row per representation |

**Statistic** the within-line share of total variance: the mean over cell lines of the within-line
variance of the cell embeddings, divided by the total variance over all eligible cells. Variance is
summed over dimensions (the trace of the covariance), which makes the ratio scale-free so `pca` and
`scgpt` sit on one axis despite sharing no units. It is one minus the intraclass correlation.

**Why it comes first.** It costs no training. If a line's cells collapse to a point in the
representation, no model of any kind can assign them different values, and every later stage measures
the wrong thing. It also separates two findings a training run alone conflates: *the input carries no
within-line structure* and *the model did not use the structure that was there*.

**Why it has no floor.** The stage reports its ratio and fails only on collapse — cells numerically
indistinguishable, tested as zero within-line variance rather than as a small one. An earlier draft
put a floor at 0.10; it had no source and was stricter than the precondition it stood for. Anything
above collapse is a matter of degree and belongs in the report as a number, not in a gate.

**Which PCA.** The stored all-cells `X_pca`, which `dataset.py` defines as the descriptive
representation. This is the one place that is correct: stage 0 is a statement about the
representation made before any training, so a fold-fitted rotation would make the answer depend on a
fold assignment that plays no part in the question. The fold-local fits exist because a *model input*
must not depend on held-out cells; nothing is fitted here.

Lines are weighted equally — a mean over lines, not over cells, so a 1,990-cell line does not count
35 times a 56-cell one.

In [3]:
def within_line_share(X, groups, lines):
    """Mean over lines of within-line variance, over the total variance. One minus the ICC.

    Variance is the trace of the covariance -- summed over dimensions -- so the ratio is invariant to
    the scale of the embedding and `pca` and `scgpt` are comparable. ddof=1 throughout: a line's cells
    are a sample of that line, not the population.

    Returns (share, per_line) so the degree is visible and not only the ratio; §2.1 reports both.
    """
    total = float(np.var(X, axis=0, ddof=1).sum())
    per_line = pd.Series(
        {ln: float(np.var(X[groups == ln], axis=0, ddof=1).sum()) for ln in lines})
    return float(per_line.mean() / total), per_line


stage0 = []
for rep in REPS:
    X = adata.obsm[rep][eligible]
    share, per_line = within_line_share(X, groups[eligible], lines_elig)
    collapsed = per_line[per_line <= 0]
    stage0.append({'rep': rep, 'n_dims': X.shape[1], 'within_line_share': share,
                   'n_lines': len(per_line), 'n_collapsed_lines': int(collapsed.size),
                   'min_line_var': float(per_line.min()), 'median_line_var': float(per_line.median()),
                   'collapse': bool(share <= 0 or collapsed.size)})

stage0 = pd.DataFrame(stage0)
stage0.to_csv(OUT / 'stage0_input_ceiling.csv', index=False)
print(stage0.to_string(index=False))
print()
for r in stage0.itertuples():
    verdict = ('COLLAPSE -- the cells of at least one line are numerically identical; no model can '
               'separate them' if r.collapse else 'no collapse')
    print(f'{r.rep:9s}  within-line share of total variance = {r.within_line_share:.3f}  -> {verdict}')
print()
print('Reported, not gated: any value above collapse is a matter of degree and belongs in the report '
      'as a number (§2.1). A split answer between the two representations is itself a result -- it '
      'locates the structure in the encoding rather than in the model.')

    rep  n_dims  within_line_share  n_lines  n_collapsed_lines  min_line_var  median_line_var  collapse
  X_pca     512           0.415833      153                  0    329.920715       606.074524     False
X_scGPT     512           0.461339      153                  0      0.018902         0.027547     False

X_pca      within-line share of total variance = 0.416  -> no collapse
X_scGPT    within-line share of total variance = 0.461  -> no collapse

Reported, not gated: any value above collapse is a matter of degree and belongs in the report as a number (§2.1). A split answer between the two representations is itself a result -- it locates the structure in the encoding rather than in the model.


### 3.3 · The runs

| | |
|---|---|
| **in** | `adata`, `PANEL`, `paths.raw_h5ad` (counts, for the per-fold PCA) |
| **out** | `mil_training_folds.csv`; `oof[(rep, seed)] → (per-cell predictions, fold log)` |

Six runs of five folds. The fold partition comes from `cv.grouped_folds` and `cv.inner_holdout`, the
same deterministic functions `4a` calls, so fold *f* holds out the same lines in both notebooks under
every seed — which is what makes stage 1's "same lines, same folds" true rather than intended.

**Why `counts_h5ad` is passed.** Under cross-validation a PCA representation is refitted inside each
fold on that fold's fitting cells: a model input may not depend on the held-out lines. The
alternative — the stored all-cells `X_pca` — is the leak that decision closed, and asking for it
requires `all_cells_pca=True` said out loud.

**Why `PCA_SEED` is fixed and is not the model seed.** With the representation held constant, stage
2's cross-seed agreement isolates *model initialisation*, which is how §2.3 words the stage. Letting
the PCA move with the seed would make stage 2 a test of the whole pipeline — stricter, but a failure
could not be attributed to the model rather than to the PCA fit. It must equal `4a`'s `PCA_SEED`, or
"seed 43" means different things in the two notebooks and stage 1 compares different inputs. Cost:
the three seeds are then independent draws of the model, not of the representation.

In [4]:
PCA_SEED = 42   # fixed across model seeds (Selin, 13.08.2026) -- see the markdown above

oof, fold_log = {}, []
for rep in REPS:
    for seed in SEEDS:
        pred, folds = bag_oof_predictions(
            adata, rep, PANEL,
            config=TrainConfig(epochs=EPOCHS, seed=seed, loss=LOSS), n_splits=N_SPLITS,
            density_weighting=ALPHA > 0, alpha=ALPHA, init_head_bias=True,
            counts_h5ad=paths.raw_h5ad, pca_seed=PCA_SEED,
            tag=f'mil_{rep}_s{seed}')
        oof[(rep, seed)] = (pred, folds)
        fold_log.extend([{**f, 'model': 'mil', 'alpha': ALPHA, 'loss': LOSS, 'seed': seed} for f in folds])
        print(f'== done {rep} seed={seed}')

fold_log = pd.DataFrame(fold_log)
fold_log.to_csv(OUT / 'mil_training_folds.csv', index=False)
print()
print(fold_log.groupby(['rep', 'seed'])[['best_epoch', 'best_val_obj']].mean().round(4).to_string())
print()
print('bags per fold (= training examples, = gradient steps per epoch):')
print(fold_log.groupby('fold')[['n_fit_bags', 'n_fit_cells', 'largest_bag']].first().to_string())

  fold 1: PCA fitted on 27516 cells -> (53513, 512)


  fold 2: PCA fitted on 28668 cells -> (53513, 512)


  fold 3: PCA fitted on 26111 cells -> (53513, 512)


  fold 4: PCA fitted on 27700 cells -> (53513, 512)


  fold 5: PCA fitted on 27137 cells -> (53513, 512)
[mil_X_pca_s42_f1] Training on device: mps | 105 train bags, 19 early-stopping bags


...

[mil_X_scGPT_s44_f5] epoch  20 | train bag MSE 0.02177 | val bag MSE 0.04546 | lr 1.00e-03


[mil_X_scGPT_s44_f5] epoch  25 | train bag MSE 0.01749 | val bag MSE 0.04288 | lr 5.00e-04


[mil_X_scGPT_s44_f5] epoch  30 | train bag MSE 0.01423 | val bag MSE 0.04204 | lr 2.50e-04


[mil_X_scGPT_s44_f5] epoch  35 | train bag MSE 0.01262 | val bag MSE 0.04073 | lr 1.25e-04


[mil_X_scGPT_s44_f5] early stop at epoch 38 (best 28, 0.03910)
== done X_scGPT seed=44

              best_epoch  best_val_obj
rep     seed                          
X_pca   42          17.2        0.0334
        43          12.4        0.0340
        44          18.8        0.0325
X_scGPT 42          15.4        0.0287
        43          18.6        0.0284
        44          18.8        0.0290

bags per fold (= training examples, = gradient steps per epoch):
      n_fit_bags  n_fit_cells  largest_bag
fold                                      
1            105        27516          898
2            103        28668         1990
3            103        26111          898
4            103        27700         1990
5            103        27137          898


### 3.4 · Persisting the predictions

| | |
|---|---|
| **out** | `runs/percell/percell_mil_*.npy` — `(53513, 11)` float32, NaN where a cell was never held out |
| | `mil_within_line_spread.csv` — one row per (rep, seed, drug, line): sd, cell count, mean |
| | `mil_oof_predictions.csv` — line-level, in `4a`'s column layout, read by `5_evaluation` |

The per-cell prediction is this model's native output and every stage below reads it; the line-level
table is the mean derived from it. Both are written in `4a`'s formats so the same scorer reads both
architectures.

`cell_index.csv` and `drug_order.json` are shared with `4a` and carry the row and column identities,
so an array is never read positionally against a differently ordered `adata`. If `4a` wrote them
first they are checked rather than overwritten: a disagreement means the two notebooks are not
indexing the same cells, which would make every cell-for-cell comparison meaningless while looking
fine.

In [5]:
index = pd.DataFrame({'cell': adata.obs_names, 'cell_line': groups})
index_csv, order_json = PERCELL / 'cell_index.csv', PERCELL / 'drug_order.json'
if index_csv.exists():
    # Written by whichever of 4a / 4b ran first. Checked, not overwritten: if the two disagree, every
    # cell-for-cell comparison below is comparing different cells and nothing downstream would say so.
    prior = pd.read_csv(index_csv)
    if not prior['cell'].equals(index['cell']):
        raise ValueError(
            f'{index_csv} lists a different cell order than this run. 4a and 4b must index the same '
            f'cells in the same order for stage 1 to compare like with like. Delete {PERCELL} and '
            f're-run both, rather than reconciling them by hand.')
    if json.loads(order_json.read_text()) != PANEL:
        raise ValueError(f'{order_json} lists a different drug order than PANEL.')
else:
    index.to_csv(index_csv, index=False)
    order_json.write_text(json.dumps(PANEL, indent=1))

spread, tidy = [], []
for (rep, seed), (pred, folds) in oof.items():
    np.save(PERCELL / f'percell_mil_{rep}_a{ALPHA:g}_s{seed}.npy', pred.astype(np.float32))
    # ddof=1: a sample spread over that line's sequenced cells, not the population it stands for.
    # The identical aggregation 4a §4 performs -- same groupby, same estimator -- because stage 1
    # compares the two columns directly and a difference in how they were computed would read as a
    # difference between the architectures.
    long = (pd.DataFrame(pred, columns=PANEL)
            .assign(cell_line=groups)
            .melt(id_vars='cell_line', var_name='drug', value_name='pred')
            .dropna(subset=['pred']))
    agg = (long.groupby(['cell_line', 'drug'], sort=False)['pred']
           .agg(n_cells='size', within_line_sd=lambda v: v.std(ddof=1), line_mean_pred='mean')
           .reset_index())
    spread.append(agg.assign(rep=rep, seed=seed))
    tidy.append(line_level_predictions(pred, adata, PANEL, folds=folds, rep=rep, model='mil',
                                       alpha=ALPHA, loss=LOSS, seed=seed))

within_mil = pd.concat(spread, ignore_index=True)[
    ['rep', 'seed', 'drug', 'cell_line', 'n_cells', 'within_line_sd', 'line_mean_pred']]
within_mil.to_csv(OUT / 'mil_within_line_spread.csv', index=False)
oof_tidy = pd.concat(tidy, ignore_index=True)
oof_tidy.to_csv(OUT / 'mil_oof_predictions.csv', index=False)

print(f'{len(oof)} runs x {pred.shape} per-cell predictions -> {PERCELL}/')
print(f'{len(within_mil)} (rep, seed, drug, line) rows -> mil_within_line_spread.csv')
print(f'{len(oof_tidy)} line-level rows -> mil_oof_predictions.csv')
print()
print('median within-line sd of per-cell predictions:')
print(within_mil.groupby(['rep', 'seed'])['within_line_sd'].median().round(5).to_string())

6 runs x (53513, 11) per-cell predictions -> runs/percell/
10098 (rep, seed, drug, line) rows -> mil_within_line_spread.csv
9810 line-level rows -> mil_oof_predictions.csv

median within-line sd of per-cell predictions:
rep      seed
X_pca    42      0.09465
         43      0.11158
         44      0.08703
X_scGPT  42      0.03336
         43      0.03842
         44      0.03964


### 3.5 · Stage 7 — the positive control

| | |
|---|---|
| **in** | `oof` per-cell predictions; line-level labels; the fold assignment |
| **out** | `stage7_positive_control.csv` — one row per (rep, seed, drug, line pair) |

No bags are materialised and no extra forward pass is made. Instance-level MIL has no attention, so a
cell's prediction does not depend on which cells accompany it — the prediction it would receive
inside the synthetic bag *is* the out-of-fold prediction already computed, and the within-bag AUROC
is a statistic of those two groups of numbers. This is a property of the architecture, not a
shortcut.

No subsampling to equal cell counts is needed either: AUROC is invariant to group sizes and
Mann–Whitney's null accommodates unequal $n$; the recovered gap is a difference of two group means.
The mixture weight enters only the bag-level diagnostic, computed from the two group means directly.

**Pairs are restricted to lines held out by the same fold.** A and B are otherwise predicted by
different models, and any systematic offset between two fold-models would enter the AUROC as if it
were signal. The cost is roughly four fifths of the candidate pairs. A control that can be passed by
inter-model offset is not a control.

**The test is one-sided** — the direction is predicted in advance, so a two-sided test would spend
half its power on an outcome the criterion does not accept.

⬜ **Open.** §2 fixes the test *per bag* and does not say how thousands of bags become one verdict.
Written here as Benjamini–Hochberg at FDR 0.05 with the surviving fraction reported. Alternatives: a
Wilcoxon signed-rank of the per-pair AUROCs against 0.5 (one test, but pairs share lines and are not
independent), or a permutation null on the median AUROC (respects the dependence, costs a
simulation).

In [6]:
SAME_FOLD_PAIRS = True   # see the OPEN note above
FDR = 0.05               # Benjamini & Hochberg, JRSS-B 57(1) 1995; the conventional level

y_lines_all, obs_lines_all = line_level(Y, M, groups, lines_elig)
y_line = pd.DataFrame(y_lines_all, index=lines_elig, columns=PANEL)
obs_line = pd.DataFrame(obs_lines_all, index=lines_elig, columns=PANEL)
line_fold = pd.Series({ln: f['fold'] for f in oof[(REPS[0], SEEDS[0])][1] for ln in f['val_lines']})
cells_of = {ln: np.flatnonzero((groups == ln) & eligible) for ln in lines_elig}


def bottom_quartile_pairs(drug):
    """Line pairs for one drug whose |y_A - y_B| falls in the bottom quartile of that drug's own
    pairwise differences, oriented so A is the more resistant line (higher label).

    The quartile is computed on ALL pairs of lines screened against the drug -- that is the drug's
    pairwise spread, and it is the reference §2.2 names. The same-fold restriction is applied
    afterwards, so it filters which of those pairs are usable and does not move the cut.
    """
    obs = obs_line[drug]
    ln = np.asarray(obs.index[obs])
    y = y_line.loc[ln, drug].to_numpy()
    i, j = np.triu_indices(len(ln), k=1)
    gap = np.abs(y[i] - y[j])
    if gap.size == 0:
        return []
    cut = np.quantile(gap, 0.25)
    keep = np.flatnonzero(gap <= cut)
    out = []
    for k in keep:
        a, b = (ln[i[k]], ln[j[k]]) if y[i[k]] >= y[j[k]] else (ln[j[k]], ln[i[k]])
        if SAME_FOLD_PAIRS and line_fold[a] != line_fold[b]:
            continue
        out.append((a, b, float(y_line.loc[a, drug] - y_line.loc[b, drug])))
    return out


PAIRS = {d: bottom_quartile_pairs(d) for d in PANEL}
print('bottom-quartile pairs per drug (after the same-fold restriction):')
print(pd.Series({d: len(v) for d, v in PAIRS.items()}).to_string())

rows = []
for (rep, seed), (pred, _) in oof.items():
    for j, drug in enumerate(PANEL):
        for a, b, gap in PAIRS[drug]:
            sa = pred[cells_of[a], j]
            sb = pred[cells_of[b], j]
            sa, sb = sa[np.isfinite(sa)], sb[np.isfinite(sb)]
            if sa.size == 0 or sb.size == 0 or gap == 0:
                continue
            # One-sided: A is the more resistant line by construction, so the predicted direction is
            # A > B. U/(n_A n_B) is the AUROC -- the probability a random A-cell outranks a random
            # B-cell -- and its p-value is the within-bag permutation null of §2.2 evaluated exactly.
            u = mannwhitneyu(sa, sb, alternative='greater')
            rows.append({'rep': rep, 'seed': seed, 'drug': drug, 'line_a': a, 'line_b': b,
                         'n_a': sa.size, 'n_b': sb.size, 'gap': gap,
                         'auroc': u.statistic / (sa.size * sb.size), 'p': u.pvalue,
                         'recovered_gap_frac': (sa.mean() - sb.mean()) / gap})

stage7 = pd.DataFrame(rows)


def bh_reject(p, q=FDR):
    """Benjamini-Hochberg step-up: boolean rejections at FDR q."""
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)
    thresh = q * np.arange(1, p.size + 1) / p.size
    passed = np.flatnonzero(p[order] <= thresh)
    out = np.zeros(p.size, dtype=bool)
    if passed.size:
        out[order[:passed[-1] + 1]] = True
    return out


stage7['significant'] = False
for key, g in stage7.groupby(['rep', 'seed'], sort=False):
    stage7.loc[g.index, 'significant'] = bh_reject(g['p'])
stage7.to_csv(OUT / 'stage7_positive_control.csv', index=False)

summary7 = (stage7.groupby(['rep', 'seed'])
            .agg(n_pairs=('auroc', 'size'), median_auroc=('auroc', 'median'),
                 frac_significant=('significant', 'mean'),
                 median_recovered=('recovered_gap_frac', 'median'),
                 median_gap=('gap', 'median'))
            .round(4))
print()
print(summary7.to_string())
print()
print('AUROC is the instrument\'s MEASURED SENSITIVITY and is reported with every downstream negative '
      '(§2.2). recovered_gap_frac is reported as a description and gates nothing -- it is '
      'calibration-sensitive, and its denominator is small by construction under bottom-quartile '
      'pairing.')
print('⚠️ A stage-7 failure ENDS THE RUN and means "Q2 unanswered, the instrument was not '
      'demonstrated" -- never "no heterogeneity found". The aggregator is not swapped for one that '
      'passes (§2, and the closing note).')

bottom-quartile pairs per drug (after the same-fold restriction):
doxorubicin    545
platin         549
etoposide      505
paclitaxel     513
gemcitabine    549
imatinib       511
erlotinib      571
sorafenib      543
dasatinib      534
crizotinib     516
afatinib       473



              n_pairs  median_auroc  frac_significant  median_recovered  median_gap
rep     seed                                                                       
X_pca   42       5809        0.5182            0.4569            0.2390      0.0219
        43       5809        0.5133            0.4431            0.2259      0.0219
        44       5809        0.5209            0.4629            0.2712      0.0219
X_scGPT 42       5809        0.5415            0.4873            0.1770      0.0219
        43       5809        0.5304            0.4794            0.1758      0.0219
        44       5809        0.5373            0.4837            0.1706      0.0219

AUROC is the instrument's MEASURED SENSITIVITY and is reported with every downstream negative (§2.2). recovered_gap_frac is reported as a description and gates nothing -- it is calibration-sensitive, and its denominator is small by construction under bottom-quartile pairing.
⚠️ A stage-7 failure ENDS THE RUN and means "Q2 un

### 3.6 · Stage 1 — spread against `4a`

| | |
|---|---|
| **in** | `mil_within_line_spread.csv`; `4a`'s `panel_within_line_spread.csv` at the same α and loss |
| **out** | `stage1_spread_vs_percell.csv` — one row per (rep, seed) |

Passes when MIL's within-line sd exceeds `4a`'s, paired on (rep, seed, drug, cell line). No margin —
§2.2 shows the bare comparison is a test of exactly the term mean pooling deletes.

The merge is **seed-matched** and asserts one-to-one: same lines, drugs, folds *and* initialisation
on both sides, so the architecture is the only thing differing between the two columns. It raises if
`4a` has not run, and there is deliberately no fallback.

⬜ **Open.** §2.2 fixes *what* is compared, not how thousands of paired values become one verdict.
Written as the paired fraction — the share of (drug, line) cells where MIL exceeds `4a`, verdict at
more than half — with the two medians and a Wilcoxon printed beside it so the choice is visible
rather than hidden in it.

In [7]:
from scipy.stats import wilcoxon

PERCELL_4A = PANEL_OUT / 'panel_within_line_spread.csv'
if not PERCELL_4A.exists():
    raise FileNotFoundError(
        f'{PERCELL_4A} not found. Stage 1 compares this model\'s within-line spread against '
        f'4a_percell_training\'s on the same lines, drugs and folds, and there is deliberately no '
        f'fallback: 4a must run first (§2.3, the dependency note). Run 4a section A, then this cell.')

within_4a = pd.read_csv(PERCELL_4A).query('alpha == @ALPHA and loss == @LOSS')
if within_4a.empty:
    raise ValueError(f'{PERCELL_4A.name} holds no rows at alpha={ALPHA}, loss={LOSS}; 4a must have swept both.')
# SEED-MATCHED since 13.08.2026, when 4a gained three seeds. Before that 4a ran one seed and the
# merge keyed on (rep, drug, cell_line) alone; with three it would have matched each MIL row to
# three 4a rows, which `validate='many_to_one'` correctly refuses. Matching seed to seed is the
# comparison stage 1 actually wants: the same lines, drugs, folds AND initialisation on both sides,
# so the only thing differing between the two columns is the architecture.
_shared = sorted(set(within_4a['seed']) & set(within_mil['seed']))
if not _shared:
    raise ValueError(
        f'4a wrote seeds {sorted(set(within_4a["seed"]))} and 4b ran {sorted(set(within_mil["seed"]))}; '
        f'stage 1 compares seed to seed, so they must overlap.')
print(f'seeds compared: {_shared}')

# ⚠️ NOT `pred_std` in panel_per_drug_correlation.csv, which is the spread of LINE-LEVEL predictions
# ACROSS cell lines -- a between-line quantity and the opposite of this one (§2.3).
paired = within_mil.merge(
    within_4a[['rep', 'seed', 'drug', 'cell_line', 'within_line_sd', 'n_cells']],
    on=['rep', 'seed', 'drug', 'cell_line'], suffixes=('_mil', '_4a'), validate='one_to_one')
if not np.array_equal(paired['n_cells_mil'], paired['n_cells_4a']):
    raise ValueError(
        'the two notebooks disagree on how many cells a (drug, cell line) contributes, so they did '
        'not hold out the same cells. Stage 1 requires matching folds (§2.3).')
print(f'{len(paired)} paired (rep, seed, drug, cell line) rows | '
      f'{paired[["drug", "cell_line"]].drop_duplicates().shape[0]} distinct (drug, line) pairs')

stage1 = (paired.assign(mil_exceeds=lambda d: d.within_line_sd_mil > d.within_line_sd_4a)
          .groupby(['rep', 'seed'])
          .apply(lambda g: pd.Series({
              'n_pairs': len(g),
              'frac_mil_exceeds': g.mil_exceeds.mean(),
              'median_sd_mil': g.within_line_sd_mil.median(),
              'median_sd_4a': g.within_line_sd_4a.median(),
              'wilcoxon_p': wilcoxon(g.within_line_sd_mil, g.within_line_sd_4a,
                                     alternative='greater').pvalue,
          }), include_groups=False)
          .round(5))
stage1['passes'] = stage1['frac_mil_exceeds'] > 0.5
stage1.to_csv(OUT / 'stage1_spread_vs_percell.csv')
print()
print(stage1.to_string())
print()
print('No margin, by construction: 4a\'s objective charges for within-line variance at full weight '
      'in every batch and the bag objective does not contain the term at all, so the bare comparison '
      'IS the test of the deleted term (§2.3).')

10098 paired (rep, seed, drug, cell line) rows | 1683 distinct (drug, line) pairs

              n_pairs  frac_mil_exceeds  median_sd_mil  median_sd_4a  wilcoxon_p  passes
rep     seed                                                                            
X_pca   42     1683.0           1.00000        0.09465       0.02822         0.0    True
        43     1683.0           1.00000        0.11158       0.02822         0.0    True
        44     1683.0           1.00000        0.08703       0.02822         0.0    True
X_scGPT 42     1683.0           0.81818        0.03336       0.02186         0.0    True
        43     1683.0           0.88948        0.03842       0.02186         0.0    True
        44     1683.0           0.85859        0.03964       0.02186         0.0    True

No margin, by construction: 4a's objective charges for within-line variance at full weight in every batch and the bag objective does not contain the term at all, so the bare comparison IS the test of the 

### 3.7 · Stage 2 — reproducibility

| | |
|---|---|
| **in** | `oof` per-cell predictions, all three seeds |
| **out** | `stage2_cross_seed_agreement.csv` — one row per (rep, seed pair, drug, line) |

For each (representation, drug, line) and each of the three seed pairs, the agreement is the Spearman
correlation across that line's cells between the two seeds' per-cell predictions.

**The shuffled-cell null needs no simulation, because it is this statistic's own null.** §2.3 defines
it as permuting cell identities within a line, which destroys cell-specific correspondence while
preserving each seed's marginal distribution — precisely the permutation null of a rank correlation,
so `spearmanr`'s p-value evaluates it directly.

Rank-based rather than Pearson, for the reason stage 7 is: it reads order, and order is what stages 1
and 2 rest on, while mean pooling gives the model a standing shrinkage incentive that a
scale-sensitive statistic would be sensitive to. Lines with fewer than three usable cells carry no
defined correlation and are counted out rather than dropped silently.

⬜ **Open.** Same aggregation question as stage 7, written the same way (BH at FDR 0.05) so the two
are read on one scale. The median ρ is printed beside it because §2.3 requires the agreement value to
be reported.

In [8]:
from itertools import combinations

MIN_CELLS_FOR_RHO = 3   # a Spearman correlation is undefined below this; not a threshold on the data

rows, skipped = [], 0
for rep in REPS:
    for s1, s2 in combinations(SEEDS, 2):
        p1, p2 = oof[(rep, s1)][0], oof[(rep, s2)][0]
        for j, drug in enumerate(PANEL):
            for ln in lines_elig:
                ci = cells_of[ln]
                a, b = p1[ci, j], p2[ci, j]
                ok = np.isfinite(a) & np.isfinite(b)
                if ok.sum() < MIN_CELLS_FOR_RHO:
                    skipped += 1
                    continue
                # One-sided: agreement means the two seeds order the cells the SAME way. The p-value
                # is the within-line shuffled-cell null of §2.4, evaluated exactly rather than
                # simulated -- permuting one vector's cell identities IS this statistic's null.
                r = spearmanr(a[ok], b[ok], alternative='greater')
                rows.append({'rep': rep, 'seed_a': s1, 'seed_b': s2, 'drug': drug, 'cell_line': ln,
                             'n_cells': int(ok.sum()), 'rho': r.statistic, 'p': r.pvalue})

stage2 = pd.DataFrame(rows)
stage2['significant'] = False
for key, g in stage2.groupby(['rep', 'seed_a', 'seed_b'], sort=False):
    stage2.loc[g.index, 'significant'] = bh_reject(g['p'])
stage2.to_csv(OUT / 'stage2_cross_seed_agreement.csv', index=False)

summary2 = (stage2.groupby(['rep', 'seed_a', 'seed_b'])
            .agg(n=('rho', 'size'), median_rho=('rho', 'median'),
                 frac_significant=('significant', 'mean'))
            .round(4))
print(f'{len(stage2)} (rep, seed pair, drug, cell line) agreements | '
      f'{skipped} skipped for fewer than {MIN_CELLS_FOR_RHO} usable cells')
print()
print(summary2.to_string())
print()
print('per representation, pooled over seed pairs:')
print(stage2.groupby('rep').agg(median_rho=('rho', 'median'),
                                frac_significant=('significant', 'mean')).round(4).to_string())
print()
print('The agreement VALUE is reported, not only the verdict (§2.4): a result that is statistically '
      'clear but small must be visible as small.')

10098 (rep, seed pair, drug, cell line) agreements | 0 skipped for fewer than 3 usable cells

                          n  median_rho  frac_significant
rep     seed_a seed_b                                    
X_pca   42     43      1683      0.2475            0.8134
               44      1683      0.2834            0.8693
        43     44      1683      0.2556            0.8295
X_scGPT 42     43      1683      0.8605            0.9917
               44      1683      0.8376            0.9857
        43     44      1683      0.8897            1.0000

per representation, pooled over seed pairs:
         median_rho  frac_significant
rep                                  
X_pca        0.2620            0.8374
X_scGPT      0.8662            0.9925

The agreement VALUE is reported, not only the verdict (§2.4): a result that is statistically clear but small must be visible as small.


### 3.8 · Stage 6 — the confound veto

| | |
|---|---|
| **in** | `oof` predictions; `obs`: `total_counts`, `Genes_expressed`, `pct_counts_mt`, `G1/S_score`, `G2/M_score` |
| **out** | `stage6_confounds.csv` — R² and adjusted R² per (rep, seed, drug, line), plus per-covariate ρ |

Regress each cell's predicted response on the four covariates, centred **within line** — the
between-line difference is what stages 1 and 2 are not about, and leaving it in would let a covariate
that merely differs between lines look like an explanation.

This is the stage that tests the strongest rival explanation. Stages 1 and 2 establish that
predictions vary within a line and that seeds agree on *which* cells; sequencing depth produces
exactly that pattern, since it varies within a line and reproduces across seeds perfectly, being a
property of the cell rather than the model. On log-CPM PCA depth is routinely a dominant axis.

**Two covariates are recovered from the raw counts.** SCP542 arrives as CPM, so library size is
divided out of every processed file, and only 4 of 13 `MT-` genes survive HVG selection. Both are
read instead from the study's un-normalised UMI matrix and joined by barcode
(`scripts/preprocessing/qc_covariates.py`), over the **full** gene set: a total over the variable
genes is not depth but depth times the share of a cell's counts falling in that set, and that share
is biological. A confounder carrying the signal under test can veto a true positive.

**The veto fires on a comparison, not a threshold** — confound R²_adj against the variance two seeds
share (ρ²). Both describe the same within-line predictions; squaring ρ puts them on one scale. If the
confounds account for as much as reproducibility does, what stages 1 and 2 measured is an artifact
that reproduces because the confounds do. No constant is chosen. The one arguable assumption: ρ² and
R²_adj are both within-line variance fractions but not the same estimator.

**Both R² and adjusted R² are reported.** Plain R² is biased upward by the five regressors and the
bias scales with 1/n, while a line contributes 56 to 1,990 cells — so a single bar on the unadjusted
figure would be a different bar for every line. The veto reads the adjusted one.

In [9]:
# The four §2.5 covariates and the obs column each resolves to. total_counts and pct_counts_mt are
# written by scripts/preprocessing/qc_covariates.py from the raw UMI matrix; Genes_expressed and the
# cell-cycle scores come from SCP542's own metadata. Cell cycle is TWO columns, G1 and G2, and both
# enter the regression -- collapsing them to one score would be a choice nobody made.
CONFOUNDS = {
    'total_counts': 'total_counts',
    'genes_detected': 'Genes_expressed',
    'mito_fraction': 'pct_counts_mt',
    # ⚠️ The columns are 'G1/S_score' and 'G2/M_score', NOT 'G1'/'G2' (fixed 13.08.2026, Gate 5).
    # The earlier names were read off h5py GROUP keys: a forward slash in a column name becomes a
    # group hierarchy in the h5 file, so `obs/G1` is a group containing `S_score` and looks like a
    # column called 'G1' to anything inspecting the file structure instead of the column-order
    # attribute. Both would have raised KeyError on the first real run.
    'cell_cycle_G1S': 'G1/S_score',
    'cell_cycle_G2M': 'G2/M_score',
}
missing = [f'{k} -> obs[{v!r}]' for k, v in CONFOUNDS.items() if v not in adata.obs.columns]
if missing:
    raise KeyError(
        'the confound veto is defined on covariates this h5ad does not carry: '
        + '; '.join(missing) + '. total_counts and pct_counts_mt are written by '
        'scripts/preprocessing/qc_covariates.py from SCP542\'s raw UMIcount_data.txt, and any file '
        'built before 13.08.2026 predates them -- re-run the convert step. This raises rather than '
        'dropping the covariate, because a veto that silently narrows itself turns a blocked check '
        'into a passed one (§3.8).')

C = adata.obs[list(CONFOUNDS.values())].to_numpy(dtype=float)


def within_line_r2(y, Xc):
    """(R^2, adjusted R^2) of the per-cell predictions on the covariates, centred within the line.

    Centring within line is what makes this a statement about WITHIN-line variation: the between-line
    difference is exactly what stages 1 and 2 are not about, and leaving it in would let a covariate
    that merely differs between lines look like an explanation.

    BOTH are returned because the choice between them changes the number a veto bar would be set on,
    and that bar is Selin's (§3.8). Plain R^2 is biased upward by the number of regressors -- five
    here -- and the bias scales with 1/n, while a cell line contributes anywhere from 56 to 1,990
    cells. So the plain figure is inflated by a different amount for every line, which is exactly the
    axis a single threshold is compared across. The adjusted figure removes that, at the cost of
    being able to go negative when the covariates explain nothing.
    """
    n, p = Xc.shape
    y = y - y.mean()
    Xc = Xc - Xc.mean(0)
    denom = float((y ** 2).sum())
    if denom <= 0 or n <= p + 1:
        return np.nan, np.nan
    beta, *_ = np.linalg.lstsq(Xc, y, rcond=None)
    r2 = 1.0 - float(((y - Xc @ beta) ** 2).sum()) / denom
    return r2, 1.0 - (1.0 - r2) * (n - 1) / (n - p - 1)


rows = []
for (rep, seed), (pred, _) in oof.items():
    for j, drug in enumerate(PANEL):
        for ln in lines_elig:
            ci = cells_of[ln]
            y = pred[ci, j]
            ok = np.isfinite(y) & np.isfinite(C[ci]).all(1)
            if ok.sum() < len(CONFOUNDS) + 2:
                continue
            r2, r2_adj = within_line_r2(y[ok], C[ci][ok])
            rec = {'rep': rep, 'seed': seed, 'drug': drug, 'cell_line': ln,
                   'n_cells': int(ok.sum()), 'r2_confounds': r2, 'r2_confounds_adj': r2_adj}
            for name, col in CONFOUNDS.items():
                rec[f'rho_{name}'] = spearmanr(y[ok], adata.obs[col].to_numpy(float)[ci][ok]).statistic
            rows.append(rec)

stage6 = pd.DataFrame(rows)
stage6.to_csv(OUT / 'stage6_confounds.csv', index=False)
# VETO_STAT is the column the veto is read on (Selin, 13.08.2026): the ADJUSTED R^2, because the
# unadjusted one is inflated by the five regressors by an amount that scales with 1/n, and a line
# contributes 56 to 1,990 cells -- so a single bar on the unadjusted figure would be a different bar
# for every line. Named once here rather than spelled out at each use, so the choice is one edit.
VETO_STAT = 'r2_confounds_adj'

cols = [VETO_STAT, 'r2_confounds'] + [f'rho_{n}' for n in CONFOUNDS]
print(f'covariates: {dict(CONFOUNDS)}')
print(f'veto is read on: {VETO_STAT} (adjusted for {len(CONFOUNDS)} regressors)')
print()
print('median over (drug, cell line), per run:')
print(stage6.groupby(['rep', 'seed'])[cols].median().round(4).to_string())
print()
print('distribution of the within-line variance explained by the confounds:')
print(stage6.groupby('rep')[[VETO_STAT, 'r2_confounds']]
      .describe()[[(VETO_STAT, '50%'), (VETO_STAT, '75%'), (VETO_STAT, 'max'),
                   ('r2_confounds', '50%'), ('r2_confounds', 'max')]].round(4).to_string())
print()
print(f'{VETO_STAT} is the veto quantity; r2_confounds is printed as a description and nothing reads '
      'it. The adjusted figure can go negative -- that reads as "the confounds explain nothing", '
      'which is the outcome this stage hopes for.')
print()
print('⬜ NO AUTOMATIC VETO YET: the bar is known to be on the adjusted R^2 but its MAGNITUDE is not '
      'set, and a permutation null cannot set it -- with hundreds of cells per line an adjusted R^2 '
      'far too small to matter is still significant. It is the one bar in §2 a null cannot replace, '
      'so the value is reported and the decision stays visible (§3.8).')

covariates: {'total_counts': 'total_counts', 'genes_detected': 'Genes_expressed', 'mito_fraction': 'pct_counts_mt', 'cell_cycle_G1S': 'G1/S_score', 'cell_cycle_G2M': 'G2/M_score'}
veto is read on: r2_confounds_adj (adjusted for 5 regressors)

median over (drug, cell line), per run:
              r2_confounds_adj  r2_confounds  rho_total_counts  rho_genes_detected  rho_mito_fraction  rho_cell_cycle_G1S  rho_cell_cycle_G2M
rep     seed                                                                                                                                 
X_pca   42              0.0556        0.0793            0.0216              0.0358             0.0507              0.0147             -0.0126
        43              0.0506        0.0759            0.0445              0.0526             0.0263              0.0061             -0.0204
        44              0.0549        0.0783            0.0298              0.0458             0.0606              0.0186              0.0062
X_scGPT

### 3.9 · The verdict

| | |
|---|---|
| **in** | `stage0`, `summary7`, `stage1`, `summary2`, `stage6` |
| **out** | `q2_verdict.csv` — one row per representation |

Assembled from the five stages in §2's order; nothing here computes a new quantity. Two rules from §2
are enforced rather than restated: a stage-7 failure ends the run and means *Q2 unanswered, the
instrument was not demonstrated*, with stages 1, 2 and 6 left unconsulted — a negative from an
instrument never shown to work is uninterpretable. And every negative is reported with stage 7's
measured AUROC attached, so a demonstrated absence is distinguishable from a blunt instrument.

In [10]:
verdict = []
for rep in REPS:
    s0 = stage0.set_index('rep').loc[rep]
    s7 = summary7.loc[rep]
    s1 = stage1.loc[rep]
    s2 = summary2.loc[rep]
    row = {
        'rep': rep,
        'stage0_within_line_share': round(float(s0.within_line_share), 4),
        'stage0_collapse': bool(s0.collapse),
        'stage7_median_auroc': round(float(s7.median_auroc.median()), 4),
        'stage7_frac_significant': round(float(s7.frac_significant.median()), 4),
        'stage1_frac_mil_exceeds': round(float(s1.frac_mil_exceeds.median()), 4),
        'stage1_passes_all_seeds': bool(s1.passes.all()),
        'stage2_median_rho': round(float(s2.median_rho.median()), 4),
        'stage2_frac_significant': round(float(s2.frac_significant.median()), 4),
        # Evaluated now that the UMI covariates exist (13.08.2026). What is still open is the
        # magnitude at which the veto FIRES, so the R^2 is carried here and read, not gated.
        'stage6_median_r2': round(float(stage6.query('rep == @rep')['r2_confounds'].median()), 4),
        'stage6_median_r2_adj': round(float(stage6.query('rep == @rep')['r2_confounds_adj'].median()), 4),
    }
    verdict.append(row)

verdict = pd.DataFrame(verdict)
verdict.to_csv(OUT / 'q2_verdict.csv', index=False)
print(verdict.to_string(index=False))
print()
for r in verdict.itertuples():
    print(f'--- {r.rep} ---')
    if r.stage0_collapse:
        print('  stage 0 COLLAPSE: the cells of a line are numerically identical in this '
              'representation. No model can separate them; later stages measure nothing.')
        continue
    print(f'  stage 0: within-line share {r.stage0_within_line_share:.3f} (reported, not gated)')
    print(f'  stage 7: median within-bag AUROC {r.stage7_median_auroc:.3f}, '
          f'{r.stage7_frac_significant:.1%} of pairs beat the permutation null')
    if r.stage7_frac_significant <= FDR:
        print('  ⛔ STAGE 7 FAILED -> THE RUN ENDS HERE. Q2 UNANSWERED, THE INSTRUMENT WAS NOT '
              'DEMONSTRATED. This is NOT "no heterogeneity found", and the aggregator is not '
              'swapped for one that passes (§2, closing note).')
        continue
    print(f'  stage 1: MIL spread exceeds 4a\'s on {r.stage1_frac_mil_exceeds:.1%} of (drug, line) '
          f'pairs; all three seeds pass: {r.stage1_passes_all_seeds}')
    print(f'  stage 2: median cross-seed rho {r.stage2_median_rho:.3f}, '
          f'{r.stage2_frac_significant:.1%} beat the shuffled-cell null')
    # D3 (Selin, 13.08.2026): the veto is a COMPARISON, not a threshold. It fires when the
    # confounds explain as much of the within-line variation as the signal reproduces. Stage 2's
    # agreement is a correlation and stage 6's is a variance fraction, so they are put on one scale
    # by squaring rho: rho^2 is the variance two seeds SHARE, R^2_adj the variance the confounds
    # EXPLAIN -- both of the same within-line per-cell predictions.
    shared = r.stage2_median_rho ** 2
    vetoed = r.stage6_median_r2_adj >= shared
    print(f'  stage 6: confound R^2_adj {r.stage6_median_r2_adj:.4f} vs the variance two seeds '
          f'share, rho^2 = {shared:.4f}')
    print(f'           -> confounds explain {100 * r.stage6_median_r2_adj / shared:.0f}% as much as '
          f'the signal reproduces')
    if vetoed:
        print('  VETO FIRES: the confounds explain at least as much of the within-line variation as '
              'reproduces across seeds, so what stages 1 and 2 measured is a technical artifact that '
              'reproduces because the confounds themselves do.')
    positive = (r.stage1_passes_all_seeds and r.stage2_frac_significant > FDR and not vetoed)
    print(f'  => Q2(a) {"POSITIVE" if positive else "NEGATIVE"} for {r.rep}, at a measured '
          f'instrument sensitivity of AUROC {r.stage7_median_auroc:.3f}.')
    print('     Q2(b) -- is this real heterogeneity of drug response -- and Q2(c) -- does it predict '
          'which cells survive -- are NOT addressed and cannot be with these measurements (§1).')

    rep  stage0_within_line_share  stage0_collapse  stage7_median_auroc  stage7_frac_significant  stage1_frac_mil_exceeds  stage1_passes_all_seeds  stage2_median_rho  stage2_frac_significant  stage6_median_r2  stage6_median_r2_adj
  X_pca                    0.4158            False               0.5182                   0.4569                   1.0000                     True             0.2556                   0.8295            0.0780                0.0539
X_scGPT                    0.4613            False               0.5373                   0.4837                   0.8586                     True             0.8605                   0.9917            0.2847                0.2656

--- X_pca ---
  stage 0: within-line share 0.416 (reported, not gated)
  stage 7: median within-bag AUROC 0.518, 45.7% of pairs beat the permutation null
  stage 1: MIL spread exceeds 4a's on 100.0% of (drug, line) pairs; all three seeds pass: True
  stage 2: median cross-seed rho 0.256, 83.0% beat the sh

## 4 · Closing analysis — what kind of cells were they?

Descriptive, and it gates nothing: by the time it runs, §2 has already decided whether the result is
positive. This section says *what was found*, not *whether* something was found — an association
discovered here cannot be promoted into evidence afterwards.

**(a)** what the predictions track — §3.8's regression shown rather than thresholded.
**(b)** which annotated programs they track — Kinker et al. scored recurrent heterogeneity programs
for this dataset independently of any drug label, and instance-level MIL gives each cell a predicted
response, so both sides are continuous and are correlated across a line's cells against a within-line
null. No top-k: correlating the full continuous signal needs no `k` to justify, uses every cell, and
is the same method as (a).
**(c)** whether the cell ordering repeats across drugs — a general axis and a drug-specific
subpopulation are different findings.

Read the enrichment carefully. Cell-cycle association is close to guaranteed and would be weak
evidence of anything: this project already refuted *the cell-line effect is largely proliferation*,
and Kinker's two named associations are on record as not transferring to this task. A program other
than cell cycle is the interesting outcome.

### 4.1 · Implementation

**(a)** is §3.8's table, already printed there and not recomputed. Note what that means given §3.8's
covariates: the part of (a) that makes the veto credible is the part that needs the recovered depth
and mitochondrial fraction.

**(b)** and **(c)** are below. Both are descriptive and gate nothing. The null for (b) is again the
within-line shuffled-cell null, evaluated exactly rather than simulated. Cell cycle is included among
the programs deliberately, so the caveat above is visible in the output rather than only in prose.

In [11]:
# Kinker's programs and the two cell-cycle scores all carry the '_score' suffix, so one
# comprehension collects them. It used to append ['G1', 'G2'] as well, which named no real column
# and double-counted the cell-cycle scores it was trying to add (fixed 13.08.2026, Gate 5).
PROGRAMS = [c for c in adata.obs.columns if c.endswith('_score')]
print(f'{len(PROGRAMS)} annotated programs: {PROGRAMS}')

rows = []
for (rep, seed), (pred, _) in oof.items():
    for j, drug in enumerate(PANEL):
        for prog in PROGRAMS:
            s = adata.obs[prog].to_numpy(dtype=float)
            for ln in lines_elig:
                ci = cells_of[ln]
                y, x = pred[ci, j], s[ci]
                ok = np.isfinite(y) & np.isfinite(x)
                if ok.sum() < MIN_CELLS_FOR_RHO:
                    continue
                r = spearmanr(y[ok], x[ok])
                rows.append({'rep': rep, 'seed': seed, 'drug': drug, 'program': prog,
                             'cell_line': ln, 'n_cells': int(ok.sum()),
                             'rho': r.statistic, 'p': r.pvalue})

programs = pd.DataFrame(rows)
programs['significant'] = False
for key, g in programs.groupby(['rep', 'seed'], sort=False):
    programs.loc[g.index, 'significant'] = bh_reject(g['p'])
programs.to_csv(OUT / 'closing_program_correlations.csv', index=False)

print()
print('median within-line rho between per-cell prediction and program score, pooled over drugs, '
      'seeds and lines:')
print(programs.groupby(['rep', 'program'])
      .agg(median_rho=('rho', 'median'), frac_significant=('significant', 'mean'))
      .round(3).unstack(0).to_string())
print()
print('⚠️ Cell-cycle association (G1, G2) is close to guaranteed and is weak evidence of anything: '
      'this project already refuted "the cell-line effect is largely proliferation", and Kinker\'s '
      'two named associations are on record as not transferring to this task (§4). A program other '
      'than cell cycle is the interesting outcome.')

12 annotated programs: ['SkinPig_score', 'EMTI_score', 'EMTII_score', 'EMTIII_score', 'IFNResp_score', 'p53Sen_score', 'EpiSen_score', 'StressResp_score', 'ProtMatu_score', 'ProtDegra_score', 'G1/S_score', 'G2/M_score']



median within-line rho between per-cell prediction and program score, pooled over drugs, seeds and lines:
                 median_rho         frac_significant        
rep                   X_pca X_scGPT            X_pca X_scGPT
program                                                     
EMTIII_score          0.018   0.054            0.160   0.458
EMTII_score           0.018   0.054            0.160   0.458
EMTI_score            0.028   0.109            0.188   0.516
EpiSen_score         -0.019  -0.017            0.179   0.475
G1/S_score            0.013   0.026            0.142   0.325
G2/M_score           -0.009  -0.040            0.264   0.551
IFNResp_score        -0.012  -0.021            0.126   0.317
ProtDegra_score      -0.064  -0.225            0.237   0.671
ProtMatu_score       -0.039  -0.115            0.119   0.469
SkinPig_score        -0.002   0.025            0.128   0.424
StressResp_score     -0.023  -0.094            0.085   0.362
p53Sen_score         -0.000   0.063    

In [12]:
# (c) Does the cell ordering repeat across drugs, or is it drug-specific? A general axis -- "these
# cells score high for everything" -- and a drug-specific subpopulation are different findings, and
# this table is the cheapest way to tell them apart. Within line, so it is a statement about the
# ordering of cells and not about the lines' differing sensitivities.
rows = []
for (rep, seed), (pred, _) in oof.items():
    for j1, j2 in combinations(range(len(PANEL)), 2):
        for ln in lines_elig:
            ci = cells_of[ln]
            a, b = pred[ci, j1], pred[ci, j2]
            ok = np.isfinite(a) & np.isfinite(b)
            if ok.sum() < MIN_CELLS_FOR_RHO:
                continue
            rows.append({'rep': rep, 'seed': seed, 'drug_a': PANEL[j1], 'drug_b': PANEL[j2],
                         'cell_line': ln, 'n_cells': int(ok.sum()),
                         'rho': spearmanr(a[ok], b[ok]).statistic})

cross_drug = pd.DataFrame(rows)
cross_drug.to_csv(OUT / 'closing_cross_drug_ordering.csv', index=False)
print('median within-line rho between the per-cell orderings of two drugs:')
print(cross_drug.groupby('rep')['rho'].describe()[['count', '25%', '50%', '75%']].round(3).to_string())
print()
print('per drug, against all others:')
both = pd.concat([cross_drug.rename(columns={'drug_a': 'drug', 'drug_b': 'other'}),
                  cross_drug.rename(columns={'drug_b': 'drug', 'drug_a': 'other'})])
print(both.groupby(['rep', 'drug'])['rho'].median().round(3).unstack(0).reindex(PANEL).to_string())
print()
print('High and uniform -> one general axis, and the per-drug heads are reading the same thing. '
      'Low -> drug-specific orderings. Descriptive: this gates nothing (§4).')

median within-line rho between the per-cell orderings of two drugs:
           count    25%    50%    75%
rep                                  
X_pca    25245.0 -0.025  0.101  0.227
X_scGPT  25245.0 -0.151  0.228  0.580

per drug, against all others:
rep          X_pca  X_scGPT
drug                       
doxorubicin  0.172    0.395
platin       0.012    0.175
etoposide    0.134    0.267
paclitaxel   0.150    0.453
gemcitabine  0.152    0.465
imatinib     0.070    0.270
erlotinib    0.093    0.189
sorafenib    0.059   -0.032
dasatinib    0.075    0.118
crizotinib   0.100    0.162
afatinib     0.113    0.150

High and uniform -> one general axis, and the per-drug heads are reading the same thing. Low -> drug-specific orderings. Descriptive: this gates nothing (§4).


### The criterion is closed

No stage may be added to §2, and none dropped, once a run exists. Building the model first and
choosing the criterion afterwards is the failure this notebook was written backwards to prevent.

That applies to the aggregator in particular. If stage 7 fails, mean pooling is **not** swapped for
something that passes: *fail → change the instrument → retry* is a forking path moved down one level,
and it would leave any subsequent positive unattributable. A stage-7 failure ends the run and means
**Q2 unanswered, the instrument was not demonstrated** — never "no heterogeneity found".